# Import Lib

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
import matplotlib.pyplot as plt
import seaborn as sns
import re
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Set Configs

In [ ]:
sns.set_palette("gray", 30)
sns.set_style("darkgrid")
cmap = "Greys"

# Import Dataset

In [ ]:
dir_path = '/kaggle/input/crime-prediction-in-chicago-in-2022/Crime Prediction in Chicago_Dataset.csv'

In [ ]:
df = pd.read_csv(dir_path)

In [ ]:
df

# Data Health

In [ ]:
class HealthCheck:
    def __init__(self, cmap="Greys"):
        self.cmap = cmap
    def msno_bar(self, data):
        return msno.bar(data)
    
    def msno_matrix(self, data):
        return msno.matrix(data)
    def msno_heatmap(self, data):
        return msno.heatmap(data)
    
    def dendrogram(self, data):

        max_text = 50
        print("-"*max_text)
        print("\033[1mDendrogram\033[0m")
        print("-"*max_text)
        text = "dendrogram The dendrogram allows you to more fully correlate variable completion, revealing trends deeper than the pairwise ones visible in the correlation heatmap:"

        lines = [text[i:i+max_text] for i in range(0, len(text), max_text)]
        for line in lines:
            print(line)
        print("-"*max_text)
        return msno.dendrogram(data)
    
    def stat(self, data):
        return data.describe().T.round(2)

    def dtype_check(self, data):
        ax = data.dtypes.value_counts().plot(kind='bar', color="grey")
        # Add numbers on top of each bar
        for p in ax.patches:
            ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='center', xytext=(0, 9), textcoords='offset points')
        plt.show()
        
    def summary_tbl(self, data: pd.DataFrame) -> pd.DataFrame:
        def get_least_most(column):
            value_counts = data[column].value_counts()
            if len(value_counts) == 0:
                return None, None
            least = value_counts.index[-1] if len(value_counts) > 1 else value_counts.index[0]
            most = value_counts.index[0]
            return least, most
        
        result = pd.DataFrame({
            "dtype": data.dtypes.values,
            "cnt_missing": data.isna().sum(),
            "ratio_missing": (data.isna().sum() / data.shape[0]) * 100,
            "n_unique": data.nunique(),
            "ratio_unique": (data.nunique() / data.shape[0]) * 100,
            "least": [get_least_most(col)[0] for col in data.columns],
            "most": [get_least_most(col)[1] for col in data.columns],
        }, index=data.columns)

        memory_usage = data.memory_usage(deep=True).sum() / 1024**2
        result['memory_usage_mb'] = memory_usage
        return result

    def summary(self, data: pd.DataFrame) -> str:
        summary_df = self.summary_tbl(data=data)
        columns = summary_df.index
        
        # Existing least and most logic
        for column in columns:
            # ... existing logic for determining least and most
            
            summary_df.loc[column, "least"] = least
            summary_df.loc[column, "most"] = most
            
        memory_usage = data.memory_usage(deep=True).sum() / 1024**2
        print(summary_df.round(2).to_markdown())
        print("\n")
        print(f"Row: {data.shape[0]}, Column: {data.shape[1]}, Memory Usage: {memory_usage:.2f} MB")


    def display(self, data: pd.DataFrame):
        display(HTML(data.to_html()))

    def missing_check(self, data):
        return pd.DataFrame({'Missing Values': data.isna().sum().sort_values(ascending=False),
                             'Percentage Missing Values': (data.isna().sum().sort_values(ascending=False)) / (data.shape[0]) * (100)}).style.background_gradient(cmap=self.cmap)
    
    def get_missing(self, data):
        return pd.DataFrame({'Missing Values': data.isna().sum().sort_values(ascending=False),
                             'Percentage Missing Values': (data.isna().sum().sort_values(ascending=False)) / (data.shape[0]) * (100)})

In [ ]:
HC = HealthCheck()

In [ ]:
HC.dtype_check(data=df)

In [ ]:
HC.summary_tbl(df)

In [ ]:
HC.missing_check(df)

# Folium 

In [ ]:
import folium
from folium.plugins import MarkerCluster
from IPython.display import display, Markdown, clear_output

In [ ]:

class FoliumMap:
    def __init__(self,
                 lat_col_name = 'Latitude', 
                 long_col_name ='Longitude',
                 popup=None):
        self.lat_col_name = lat_col_name
        self.long_col_name = long_col_name
        self.popup_col = popup
    
    def geographlize(self, data, column, target_value, n_sample, tag):
        # Filter
        transactions = data[data[column] == target_value]
        
        # Number of samples to take
        available_count = transactions.shape[0]
        sample_count = min(n_sample, available_count)  
        
        # Randomly sample 
        transactions = transactions.sample(sample_count, replace=False)

        row_count, col_count = transactions.shape
        display(Markdown(f"# **{tag}**"))
        display(Markdown(f"**- Data has {row_count} rows and {col_count} columns**"))
        display(Markdown(f"**- Latitude: {transactions[self.lat_col_name].mean()}**"))
        display(Markdown(f"**- Longitude: {transactions[self.long_col_name].mean()}**"))
        
        # Create a Folium map 
        m = folium.Map(location=[transactions[self.lat_col_name].mean(), transactions[self.long_col_name].mean()], zoom_start=12)
        
        # Create a MarkerCluster
        marker_cluster = MarkerCluster().add_to(m)
        
        # Add markers for each fraudulent transaction to the cluster
        for _, row in transactions.iterrows():
            folium.Marker(
                location=[row[self.lat_col_name], row[self.long_col_name]],
                popup=f"{self.popup_col}: {row[self.popup_col]}",
                icon=folium.Icon(color='red', icon='exclamation-sign')  # Red markers for fraud
            ).add_to(marker_cluster)
        
        # Display the map
        return m
    
    def geographlize(self, data, column, target_value, n_sample, tag):
        # Filter
        transactions = data[data[column] == target_value]
    
        # Remove rows with NaN values in the latitude or longitude columns
        transactions = transactions.dropna(subset=[self.lat_col_name, self.long_col_name])
    
        # Number of samples to take
        available_count = transactions.shape[0]
        sample_count = min(n_sample, available_count)  
    
        # Randomly sample 
        transactions = transactions.sample(sample_count, replace=False)
    
        row_count, col_count = transactions.shape
        display(Markdown(f"# **{tag}**"))
        display(Markdown(f"**- Data has {row_count} rows and {col_count} columns**"))
        display(Markdown(f"**- Latitude: {transactions[self.lat_col_name].mean()}**"))
        display(Markdown(f"**- Longitude: {transactions[self.long_col_name].mean()}**"))
        
        # Create a Folium map 
        m = folium.Map(location=[transactions[self.lat_col_name].mean(), transactions[self.long_col_name].mean()], zoom_start=12)
        
        # Create a MarkerCluster
        marker_cluster = MarkerCluster().add_to(m)
        
        # Add markers for each fraudulent transaction to the cluster
        for _, row in transactions.iterrows():
            folium.Marker(
                location=[row[self.lat_col_name], row[self.long_col_name]],
                popup=f"{self.popup_col}: {row[self.popup_col]}",
                icon=folium.Icon(color='red', icon='exclamation-sign')  # Red markers for fraud
            ).add_to(marker_cluster)
        
        # Display the map
        return m


In [ ]:
FM = FoliumMap(popup='Description')

In [ ]:
FM.geographlize(df, 
                'Arrest', 
                True, 
                10000, 
                'Arrest = True')